# Papagun — загрузка Excel через pandas

Выполняйте ячейки сверху вниз в Jupyter или редакторе с поддержкой notebooks. Нужны `pandas`, `openpyxl` и Python-ядро. Для отдельного Jupyter Notebook при необходимости установите их командой `python -m pip install pandas openpyxl notebook`.

Исходные книги только читаются. Содержимое таблиц не выводится автоматически; перед сохранением ноутбука очищайте результаты ячеек, если выводили данные.

In [ ]:
from pathlib import Path

import pandas as pd

## Исходные книги

Читаются `NL_Кол. Бюджет.xlsx` и `NL_Фин. Бюджет.xlsx` из папки `Исходники`.
Запускайте ядро из корня проекта; при другом расположении измените `DATA_DIR` на папку с этими книгами.


In [ ]:
DATA_DIR = Path.cwd() / 'Исходники'
excel_files = {
    'financial': DATA_DIR / 'NL_Фин. Бюджет.xlsx',
    'quantity': DATA_DIR / 'NL_Кол. Бюджет.xlsx',
}

missing_files = [key for key, path in excel_files.items() if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        'Не найдены книги: ' + ', '.join(missing_files)
        + '. Проверьте DATA_DIR и имена файлов.'
    )


## Загрузка всех листов

`sheet_name=None` возвращает словарь листов. `header=None` сохраняет все строки, поскольку строка заголовков ещё не согласована. `dtype=object` отключает подбор типа для всей колонки; `keep_default_na=False` сохраняет текстовые маркеры вроде `NA`, а пустые ячейки обычно представлены пустой строкой. Пропуски не заменяются нулями.

Это первичное чтение значений, а не полная модель Excel: оформление и объединения не сохраняются в DataFrame, ошибки ячеек могут стать пропусками. Формулы не пересчитываются — читаются сохранённые Excel результаты, которые могут отсутствовать или быть устаревшими. Эти случаи требуют отдельной проверки перед расчётами.

In [ ]:
workbooks = {
    key: pd.read_excel(
        path,
        sheet_name=None,
        header=None,
        dtype=object,
        keep_default_na=False,
        engine='openpyxl',
    )
    for key, path in excel_files.items()
}

financial_sheets = workbooks['financial']
quantity_sheets = workbooks['quantity']

In [ ]:
sheet_summary = pd.DataFrame([
    {'book': key, 'sheet': sheet, 'rows': frame.shape[0], 'columns': frame.shape[1]}
    for key, sheets in workbooks.items()
    for sheet, frame in sheets.items()
])
sheet_summary

## Отдельные DataFrame филиалов двух книг NL

`quantity_df` — количественный бюджет, `financial_df` — финансовый бюджет.
Для каждой книги применяется прежняя методика: одна строка исходной статьи × филиал × месяц,
шесть колонок: филиал, статья, месяц, утверждено, факт по норме, факт по 1С.
Книги не объединяются, показатели не пересчитываются и не округляются.

Используются те же 20 листов филиалов, строки 9–141 количественной книги и 9–193 финансовой.
Заголовки статей (C6), месяцы (строка 6) и три показателя (строка 7) проверяются.
Сводные и вспомогательные листы, АУП, кварталы и годовые колонки исключены.
Итоги и повторяющиеся статьи сохраняются; пустые названия пропускаются.
Пустые показатели представлены `pd.NA`, нули сохранены, ошибки Excel восстановлены как текстовые коды.
Формулы не пересчитываются: используются сохранённые результаты Excel.


In [ ]:
BRANCH_SHEETS = [
    'Актобе', 'Актобе АМЦ', 'Астана Биржан Сал', 'Астана Сарыарка',
    'Атырау', 'Атырау Нурсая', 'Атырау Нурсая 2', 'Караганда ЦАД',
    'Караганда Строителей', 'Кентау', 'Костанай', 'Лисаковск', 'Рудный',
    'Павлодар Дюсенова', 'Павлодар Павлова', 'Экибастуз', 'Петропавловск',
    'Усть-Каменогорск (NL UK)', 'Шымкент', 'Шымкент Абая',
]
ARTICLE_ROWS = {'financial': (9, 193), 'quantity': (9, 141)}  # номера строк Excel, включительно
MONTHS = [
    'январь', 'февраль', 'март', 'апрель', 'май', 'июнь',
    'июль', 'август', 'сентябрь', 'октябрь', 'ноябрь', 'декабрь',
]
RESULT_COLUMNS = [
    'Наименование филиала', 'Наименование статей', 'Месяц',
    'Утверждено', 'Факт по норме', 'Факт по 1С',
]


In [ ]:
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter


def header_text(value):
    # Нормализация применяется только к заголовкам для их проверки.
    return ' '.join(str(value).split()).casefold()


def build_branches_dataframe(source_book):
    if source_book not in ARTICLE_ROWS:
        raise ValueError("Источник должен быть 'financial' или 'quantity'.")
    sheets = workbooks[source_book]
    missing = [sheet for sheet in BRANCH_SHEETS if sheet not in sheets]
    if missing:
        raise ValueError('Не найдены листы филиалов: ' + ', '.join(missing))

    first_row, last_row = ARTICLE_ROWS[source_book]
    records, origins, issues = [], [], []
    # pandas превращает ошибки ячеек Excel в NaN. Второе чтение позволяет
    # восстановить коды ошибок, не смешивая их с пустыми значениями.
    cached_book = load_workbook(excel_files[source_book], read_only=True, data_only=True)
    try:
        for branch in BRANCH_SHEETS:
            frame = sheets[branch]
            if frame.shape[0] < last_row or frame.shape[1] < 56:
                raise ValueError(f'Недостаточный размер основной таблицы: {branch}')
            if header_text(frame.iat[5, 2]) != 'наименование статей':
                raise ValueError(f'Не найден заголовок статей в C6: {branch}')

            month_columns = {}
            for column, value in enumerate(frame.iloc[5]):
                month = header_text(value)
                if month not in MONTHS:
                    continue
                if month in month_columns:
                    raise ValueError(f'Повторяющийся заголовок месяца: {branch}, {month}')
                expected = ['утвержд.', 'факт по норме', 'факт по 1с']
                actual = [header_text(frame.iat[6, column + offset]) for offset in range(3)]
                if actual != expected:
                    raise ValueError(f'Неожиданные показатели месяца: {branch}, {month}')
                month_columns[month] = column
            if set(month_columns) != set(MONTHS):
                raise ValueError(f'Ожидаются все 12 месяцев: {branch}')

            cached_rows = cached_book[branch].iter_rows(
                min_row=first_row, max_row=last_row, max_col=frame.shape[1],
            )
            for excel_row, cells in enumerate(cached_rows, start=first_row):
                article = frame.iat[excel_row - 1, 2]
                if cells[2].data_type == 'e':
                    raise ValueError(f'Ошибка Excel вместо названия статьи: {branch}, C{excel_row}')
                if pd.isna(article) or article == '' or (isinstance(article, str) and not article.strip()):
                    continue
                if not isinstance(article, str):
                    raise ValueError(f'Название статьи не является текстом: {branch}, C{excel_row}')
                for month in MONTHS:
                    column = month_columns[month]
                    values = []
                    coordinates = []
                    for offset, metric in enumerate(RESULT_COLUMNS[3:]):
                        coordinate = f'{get_column_letter(column + offset + 1)}{excel_row}'
                        coordinates.append(coordinate)
                        cell = cells[column + offset]
                        value = frame.iat[excel_row - 1, column + offset]
                        if cell.data_type == 'e':
                            value = cell.value
                            issues.append({
                                'Строка DataFrame': len(records), 'Филиал': branch,
                                'Месяц': month, 'Показатель': metric,
                                'Ячейка': coordinate, 'Ошибка Excel': value,
                            })
                        elif pd.isna(value) or value == '':
                            value = pd.NA
                        values.append(value)
                    records.append([branch, article, month, *values])
                    origins.append([source_book, branch, excel_row, *coordinates])
    finally:
        cached_book.close()

    result = pd.DataFrame(records, columns=RESULT_COLUMNS, dtype=object)
    provenance = pd.DataFrame(origins, columns=[
        'Книга', 'Лист', 'Строка Excel',
        'Ячейка Утверждено', 'Ячейка Факт по норме', 'Ячейка Факт по 1С',
    ])
    diagnostics = pd.DataFrame(issues, columns=[
        'Строка DataFrame', 'Филиал', 'Месяц', 'Показатель', 'Ячейка', 'Ошибка Excel',
    ])
    return result, provenance, diagnostics


quantity_df, quantity_provenance, quantity_diagnostics = build_branches_dataframe('quantity')
financial_df, financial_provenance, financial_diagnostics = build_branches_dataframe('financial')

# Совместимость с прежними ячейками исследования количественного бюджета.
branches_df = quantity_df
branches_provenance = quantity_provenance
branches_diagnostics = quantity_diagnostics


In [ ]:
# Только размеры и диагностика, без значений показателей.
dataframe_summary = pd.DataFrame([
    {
        'Книга': key,
        'Листов': len(workbooks[key]),
        'Строк': len(frame),
        'Филиалов': frame['Наименование филиала'].nunique(),
        'Месяцев': frame['Месяц'].nunique(),
        'Ошибок Excel': len(diagnostics),
    }
    for key, frame, diagnostics in [
        ('quantity', quantity_df, quantity_diagnostics),
        ('financial', financial_df, financial_diagnostics),
    ]
])
dataframe_summary


## Доступ к результатам

Основные таблицы: `quantity_df` и `financial_df`.
Координаты источников с тем же индексом: `quantity_provenance` и `financial_provenance`.
Ошибки Excel: `quantity_diagnostics` и `financial_diagnostics`.
Все исходные листы доступны отдельно в `quantity_sheets` и `financial_sheets`.

`branches_df`, `branches_provenance`, `branches_diagnostics` оставлены как ссылки на количественный бюджет.
Для локального просмотра можно выполнить `quantity_df.head()` или `financial_df.head()` в отдельной ячейке.
Перед сохранением удаляйте результаты с содержимым таблиц.
Не суммируйте итоговые строки вместе с детализацией без согласованного правила.
При смене версии книг перепроверьте список филиалов и границы `ARTICLE_ROWS`.


## Аналог второго слайда: пациенты
Подтверждённый период: сентябрь 2026 года, численность на конец 9 месяцев по 18 центрам исходного слайда. План — «Утверждено», факт — «Факт по 1С», строка Excel 14; месяцы не суммируются. Статья и состав центров для дальнейшего продукта требуют подтверждения. В образце заголовок за 9 месяцев расходится с проверенными июньскими значениями, а сводка — с суммой столбцов.
Отдельный PPTX содержит редактируемую диаграмму; итог рассчитывается по показанным центрам. Исходники не изменяются. Повторная запись в существующий результат блокируется: выберите новое имя.

In [ ]:
from slide2_export import build_slide2

slide2_path = build_slide2(
    quantity_df, quantity_provenance,
    month='сентябрь',
    output='output/NL_пациенты_2026_слайд2.pptx',
)
